        # Simulación de procesos y fenómenos de ingeniería

        **Modelación y Simulación Computacional** · Maestría en Ingeniería ·
        Universidad de Sucre · periodo 2026-2

        **Unidad 3.** Simulación de sistemas y análisis de escenarios ·
        **Subtema del plan 3.1**

        Autor, Prof. Daniel Otero Meza, Ing., Ph.D.

        <!-- ENLACE_COLAB -->
        [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/<usuario>/<repositorio>/blob/main/03_cuadernos/Unidad3/U3_01_simulacion_de_procesos.ipynb)

        Este cuaderno recorre los tres regímenes de simulación que organiza la
Figura 3.2 del libro, el estacionario, el dinámico de parámetros
concentrados y el distribuido. De cada uno se ejecuta un caso completo,
con el método de solución, el criterio de parada y la verificación que
permite creerle al resultado. No se repite la teoría del capítulo, se
ejecuta.

        ## Objetivos de aprendizaje

        Al terminar este cuaderno el estudiante debe ser capaz de

        1. Plantear el residuo de un modelo estacionario según la Definición 3.1 y resolverlo con el algoritmo de Brent, con bisección, con punto fijo y con Newton, comparando iteraciones y evaluaciones.
2. Estimar el orden de convergencia observado de un método y contrastarlo con el Teorema 3.1.
3. Resolver un sistema estacionario acoplado con `scipy.optimize.root` suministrando el jacobiano analítico, y verificar la solución con un balance global que cierre a precisión de máquina.
4. Integrar un problema de valor inicial con `scipy.integrate.solve_ivp`, localizar un evento y verificar el resultado con un invariante exacto.
5. Armar un método de líneas según la Definición 3.4 y verificar el perfil contra la solución analítica.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de modo que
el cuaderno abre igual en Google Colab y en JupyterLab. La segunda fija la
semilla del curso, la paleta del libro y las funciones auxiliares. La semilla
vale 20262 y ningún resultado depende de una ejecución concreta.

In [ ]:
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict) -> None:
    """Instala solo los paquetes que no estén disponibles."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

COLORES = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.figsize": (9.0, 4.4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "legend.frameon": False})

trapecio = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def carpeta_datos() -> Path:
    """Ubica la carpeta datos sin usar rutas absolutas.

    Busca hacia arriba desde el directorio de trabajo, de modo que funcione
    tanto en el repositorio como en una sesión de Colab donde el cuaderno se
    abre suelto. Si no la encuentra, la crea junto al cuaderno.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        if (candidata / "datos").is_dir():
            return candidata / "datos"
    destino = base / "datos"
    destino.mkdir(exist_ok=True)
    return destino


def leer_datos(nombre: str, respaldo) -> pd.DataFrame:
    """Lee un archivo de datos y lo reconstruye si no está disponible.

    El argumento respaldo es una función sin argumentos que devuelve el
    mismo cuadro de datos, construido con las cifras publicadas en el libro.
    Así el cuaderno nunca depende de una descarga.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        ruta = candidata / "datos" / nombre
        if ruta.exists():
            return pd.read_csv(ruta)
    tabla = respaldo()
    tabla.to_csv(carpeta_datos() / nombre, index=False)
    return tabla


def comparar(etiqueta: str, calculado: float, libro: float,
             tol: float, unidad: str = "") -> bool:
    """Imprime y verifica un valor calculado frente al que publica el libro."""
    dif = abs(calculado - libro)
    ok = dif <= tol
    marca = "coincide" if ok else "NO coincide"
    print(f"{etiqueta:<46s} calculado {calculado:>14.6g} {unidad:<12s}"
          f" libro {libro:>12.6g}   {marca}")
    return ok


print("semilla del curso", SEMILLA)

In [ ]:
def respaldo_valores_libro() -> pd.DataFrame:
    """Cifras publicadas en el capítulo 3, transcritas del libro."""
    filas = [
    ("colebrook_velocidad", 1.6977, "m/s", "Ejemplo 3.1"),
    ("colebrook_reynolds", 507267.0, "adimensional", "Ejemplo 3.1"),
    ("colebrook_rugosidad_relativa", 0.0008667, "adimensional", "Ejemplo 3.1"),
    ("colebrook_factor_friccion", 0.0196228, "adimensional", "Ejemplo 3.1"),
    ("colebrook_perdida_carga", 8.167, "m", "Ejemplo 3.1"),
    ("colebrook_swamee_jain", 0.019742, "adimensional", "Ejemplo 3.1"),
    ("colebrook_orden_newton", 2.0, "adimensional", "Ejemplo 3.1"),
    ("lagunas_perfil_1", 142.42, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_2", 83.4, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_3", 41.5, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_4", 17.65, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_5", 7.33, "mg/L", "seccion 3.1.2"),
    ("lagunas_remocion", 97.07, "por ciento", "seccion 3.1.2"),
    ("lagunas_retencion", 8.33, "d", "seccion 3.1.2"),
    ("lagunas_carga_afluente", 300000.0, "mg/d", "seccion 3.1.2"),
    ("lagunas_carga_efluente", 8792.84, "mg/d", "seccion 3.1.2"),
    ("lagunas_consumo", 291207.16, "mg/d", "seccion 3.1.2"),
    ("fermentador_tiempo_25C", 17.0604614, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_30C", 10.9186953, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_35C", 7.5824273, "h", "Ejemplo 3.2"),
    ("fermentador_invariante", 12.5, "g/L", "Ejemplo 3.2"),
    ("fermentador_mumax_25C", 0.192, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_30C", 0.3, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_35C", 0.432, "1/h", "Ejemplo 3.2"),
    ("fermentador_evaluaciones", 584.0, "evaluaciones", "Ejemplo 3.2"),
    ("tolerancia_tiempo_rtol3", 10.9028, "h", "seccion 3.2.1"),
    ("tolerancia_error_rtol3", 0.00146, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol3", 44.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol6", 1.85e-07, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol6", 194.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol9", 2.45e-10, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol9", 584.0, "evaluaciones", "seccion 3.2.1"),
    ("circuito_autovalor_rapido", -1005.0002, "1/s", "Ejemplo 3.3"),
    ("circuito_autovalor_lento", -0.0497512, "1/s", "Ejemplo 3.3"),
    ("circuito_razon_rigidez", 20200.0, "adimensional", "Ejemplo 3.3"),
    ("circuito_pasos_rk45", 30376.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_rk45", 212576.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_pasos_bdf", 144.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_bdf", 292.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_paso_medio_rk45", 0.003292, "s", "Ejemplo 3.3"),
    ("circuito_tau_rapida", 0.000995, "s", "Ejemplo 3.3"),
    ("circuito_tau_lenta", 20.1, "s", "Ejemplo 3.3"),
    ("circuito_error_rk45", 9.1e-07, "adimensional", "Ejemplo 3.3"),
    ("circuito_error_bdf", 1.1e-06, "adimensional", "Ejemplo 3.3"),
    ("circuito_producto_h_lambda", 3.31, "adimensional", "Ejemplo 3.3"),
    ("rio_peclet_celda", 0.583, "adimensional", "Ejemplo 3.4"),
    ("rio_pico_analitico", 1.8655, "mg/L", "Ejemplo 3.4"),
    ("rio_abscisa_pico", 2460.0, "m", "Ejemplo 3.4"),
    ("rio_error_maximo", 7.18e-06, "kg/m3", "Ejemplo 3.4"),
    ("rio_masa_remanente", 24.740935, "kg", "Ejemplo 3.4"),
    ("rio_orden_observado", 2.0, "adimensional", "Ejemplo 3.4"),
    ("rio_paso_difusion", 16.67, "s", "seccion 3.3.2"),
    ("rio_paso_adveccion", 57.14, "s", "seccion 3.3.2"),
    ("rio_error_explicito_d045", 6.3e-05, "kg/m3", "Ejemplo 3.4"),
    ("riego_frontera_bruto_p045_d45", 393.8, "mm", "Ejemplo 3.5"),
    ("riego_frontera_bruto_p085_d65", 243.8, "mm", "Ejemplo 3.5"),
    ("riego_deficit_p085_d65", 21.5, "por ciento", "Ejemplo 3.5"),
    ("riego_no_dominadas_clima_normal", 5.0, "alternativas", "Ejemplo 3.5"),
    ("biogas_desviacion_replicas_mc", 0.933, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion_replicas_lhs", 0.112, "kW h/d", "Ejemplo 3.6"),
    ("biogas_reduccion_varianza", 69.0, "veces", "Ejemplo 3.6"),
    ("riego_agua_aprovechable", 126.0, "mm", "Ejemplo 3.5"),
    ("biogas_media", 92.34, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion", 21.22, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p05", 61.71, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p50", 90.05, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p95", 130.62, "kW h/d", "Ejemplo 3.6"),
    ("biogas_excedencia_110", 0.1945, "adimensional", "Ejemplo 3.6"),
    ("biogas_error_estandar", 0.15, "kW h/d", "Ejemplo 3.6"),
    ("biogas_valores_centrales", 91.53, "kW h/d", "Ejemplo 3.6"),
    ("lcoe_crf", 0.101806, "1/a", "Ejemplo 3.7"),
    ("lcoe_factor_degradacion", 0.931205, "adimensional", "Ejemplo 3.7"),
    ("lcoe_produccion_especifica", 1325.6, "kW h/(kW a)", "Ejemplo 3.7"),
    ("lcoe_nominal", 0.083523, "USD/(kW h)", "Ejemplo 3.7"),
    ("lcoe_elasticidad_inversion", 0.87355, "adimensional", "Ejemplo 3.7"),
    ("lcoe_elasticidad_tasa", 0.637, "adimensional", "Ejemplo 3.7"),
    ("lcoe_amplitud_tasa", 35.8, "por ciento", "Ejemplo 3.7"),
    ("lcoe_amplitud_irradiacion", 16.1, "por ciento", "Ejemplo 3.7"),
    ("ishigami_s1", 0.3138, "adimensional", "seccion 3.6.2"),
    ("ishigami_s2", 0.4423, "adimensional", "seccion 3.6.2"),
    ("ishigami_s3", -0.0001, "adimensional", "seccion 3.6.2"),
    ("ishigami_st3", 0.2436, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_s_B0", 0.616, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_suma_primer_orden", 0.985, "adimensional", "seccion 3.6.2"),
    ("morris_evaluaciones", 210.0, "evaluaciones", "seccion 3.6.2"),
    ]
    return pd.DataFrame(filas, columns=["clave", "valor", "unidad", "referencia"])


LIBRO = leer_datos("valores_libro_cap3.csv",
                   respaldo_valores_libro).set_index("clave")["valor"]
print(f"cifras del libro disponibles, {LIBRO.size} registros")

## 1. Del balance al residuo

La Definición 3.1 del libro llama residuo a la función que resulta de
pasar todos los términos del balance a un mismo miembro, de modo que la
solución del modelo es el vector que la anula. Cada componente conserva
las unidades del balance del cual proviene, razón por la cual un sistema
con balances de masa y de energía exige escalar antes de tomar una norma.

El Ejemplo 3.1 pide el factor de fricción de una conducción de hierro
dúctil de \(D=0.30\) m y rugosidad absoluta de 0.26 mm que transporta
0.12 m³/s de agua a 20 °C, con viscosidad cinemática de
1.004 × 10⁻⁶ m²/s. La ecuación de Colebrook y White es implícita en el
factor de fricción y no admite despeje, de manera que hay que resolverla.

In [ ]:
DIAMETRO = 0.30          # m
RUGOSIDAD = 0.26e-3      # m
CAUDAL = 0.12            # m3/s
VISCOSIDAD = 1.004e-6    # m2/s
LONGITUD = 850.0         # m
GRAVEDAD = 9.81          # m/s2

area = np.pi * DIAMETRO**2 / 4
velocidad = CAUDAL / area                     # m/s
reynolds = velocidad * DIAMETRO / VISCOSIDAD  # adimensional
rugosidad_relativa = RUGOSIDAD / DIAMETRO     # adimensional

ok = [comparar("velocidad media", velocidad, LIBRO["colebrook_velocidad"],
               5e-5, "m/s"),
      comparar("numero de Reynolds", reynolds, LIBRO["colebrook_reynolds"],
               1.0, ""),
      comparar("rugosidad relativa", rugosidad_relativa,
               LIBRO["colebrook_rugosidad_relativa"], 1e-6, "")]
assert all(ok), "revise el cálculo de la velocidad o del número de Reynolds"

### Ejercicio 1

Escriba el residuo de la ecuación de Colebrook y White tal como aparece
en el Listado 3.1 del libro. Recuerde que el residuo se obtiene pasando
el miembro derecho al izquierdo, de modo que la expresión se anula en la
solución. La celda de partida trae un residuo evidentemente falso, cuya
raíz vale 0.05, para que el cuaderno siga ejecutándose.

In [ ]:
# COMPLETE: escriba la forma residual de Colebrook-White,
#   1/sqrt(f) + 2*log10(rug/3.7 + 2.51/(Re*sqrt(f)))
# Cambie REVISAR a True cuando termine.
REVISAR = False


def residuo_colebrook(f: float, Re: float, rug: float) -> float:
    """Forma residual de la ecuación de Colebrook-White."""
    return f - 0.05        # marcador de posición, no es el residuo

In [ ]:
from scipy.optimize import brentq

f_brent = brentq(residuo_colebrook, 0.005, 0.10,
                 args=(reynolds, rugosidad_relativa), xtol=1e-14)
perdida = f_brent * LONGITUD / DIAMETRO * velocidad**2 / (2 * GRAVEDAD)

comparar("factor de fricción con Brent", f_brent,
         LIBRO["colebrook_factor_friccion"], 1e-6, "")
comparar("pérdida de carga en 850 m", perdida,
         LIBRO["colebrook_perdida_carga"], 5e-3, "m")

if REVISAR:
    assert abs(f_brent - LIBRO["colebrook_factor_friccion"]) < 1e-6, \
        "el residuo no reproduce el factor de fricción del Ejemplo 3.1"
    assert abs(perdida - LIBRO["colebrook_perdida_carga"]) < 5e-3, \
        "revise la fórmula de Darcy y Weisbach"
    print("verificación superada")
else:
    print("complete la celda anterior y ponga REVISAR = True")

## 2. Los tres métodos escalares y su costo

La Tabla 3.1 del libro compara bisección, punto fijo, secante, Newton y
Brent por su orden, por las evaluaciones que gastan en cada iteración y
por su falla característica. Aquí se implementan los tres primeros sobre
el mismo residuo y se cuentan iteraciones y evaluaciones, que es la única
manera honesta de comparar el costo. El punto fijo se aplica sobre la
variable \(x = 1/\sqrt{f}\), reescritura que hace convergente la
sustitución.

In [ ]:
def biseccion(F, a: float, b: float, tol: float = 1e-10, maxit: int = 100):
    """Bisección sobre un intervalo con cambio de signo."""
    Fa, Fb = F(a), F(b)
    if Fa * Fb > 0:
        raise ValueError("el intervalo no encierra un cambio de signo")
    evaluaciones, iteraciones, historial = 2, 0, []
    while (b - a) / 2 > tol and iteraciones < maxit:
        c = 0.5 * (a + b)
        Fc = F(c)
        evaluaciones += 1
        iteraciones += 1
        historial.append(c)
        if Fa * Fc < 0:
            b = c
        else:
            a, Fa = c, Fc
    return 0.5 * (a + b), iteraciones, evaluaciones, historial


def punto_fijo(g, x0: float, tol: float = 1e-10, maxit: int = 100):
    """Iteración de punto fijo sobre x = g(x)."""
    x, evaluaciones, historial = x0, 0, []
    for k in range(1, maxit + 1):
        x_nuevo = g(x)
        evaluaciones += 1
        historial.append(x_nuevo)
        if abs(1 / x_nuevo**2 - 1 / x**2) < tol:
            return x_nuevo, k, evaluaciones, historial
        x = x_nuevo
    return x, maxit, evaluaciones, historial


def newton(F, dF, x0: float, tol: float = 1e-10, maxit: int = 50):
    """Newton-Raphson con criterio de parada sobre el paso."""
    x, evaluaciones, historial = x0, 0, []
    for k in range(1, maxit + 1):
        paso = -F(x) / dF(x)
        evaluaciones += 2
        x = x + paso
        historial.append(x)
        if abs(paso) < tol:
            return x, k, evaluaciones, historial
    return x, maxit, evaluaciones, historial


coef_a = rugosidad_relativa / 3.7
coef_b = 2.51 / reynolds
residuo = lambda f: residuo_colebrook(f, reynolds, rugosidad_relativa)
derivada = lambda f: (-0.5 * f**-1.5
                      - coef_b * f**-1.5 / (np.log(10) * (coef_a + coef_b / np.sqrt(f))))
sustitucion = lambda x: -2 * np.log10(coef_a + coef_b * x)

f_bis, it_bis, ev_bis, hist_bis = biseccion(residuo, 0.005, 0.10)
x_pf, it_pf, ev_pf, hist_pf = punto_fijo(sustitucion, 1 / np.sqrt(0.02))
f_new, it_new, ev_new, hist_new = newton(residuo, derivada, 0.02)

resumen = pd.DataFrame(
    [("bisección", f_bis, it_bis, ev_bis),
     ("punto fijo", 1 / x_pf**2, it_pf, ev_pf),
     ("Newton-Raphson", f_new, it_new, ev_new),
     ("Brent", f_brent, np.nan, np.nan)],
    columns=["método", "factor de fricción", "iteraciones", "evaluaciones"])
print(resumen.to_string(index=False))

In [ ]:
conteos = {"bisección": (29, 31), "punto fijo": (5, 5), "Newton-Raphson": (4, 8)}
for metodo, (it_libro, ev_libro) in conteos.items():
    fila = resumen.loc[resumen["método"] == metodo].iloc[0]
    comparar(f"{metodo}, iteraciones", fila["iteraciones"], it_libro, 0, "")
    comparar(f"{metodo}, evaluaciones", fila["evaluaciones"], ev_libro, 0, "")

if REVISAR:
    assert int(resumen.loc[0, "iteraciones"]) == 29
    assert int(resumen.loc[1, "iteraciones"]) == 5
    assert int(resumen.loc[2, "iteraciones"]) == 4
    print("los tres conteos coinciden con el Ejemplo 3.1")

### Ejercicio 2

El orden de convergencia se estima con tres iteraciones consecutivas,
mediante el cociente de logaritmos de errores sucesivos,
\(\hat{p} = \ln(e_{k+1}/e_{k}) / \ln(e_{k}/e_{k-1})\). Complete la
función y compruebe que Newton entrega un orden próximo a dos, tal como
anuncia el Teorema 3.1, mientras que la bisección se queda en uno.

In [ ]:
# COMPLETE: estime el orden de convergencia con los tres primeros errores.
REVISAR_ORDEN = False


def orden_convergencia(errores) -> float:
    """Orden observado a partir de tres errores consecutivos."""
    e = np.asarray(errores, dtype=float)
    return 1.0            # marcador de posición

In [ ]:
f_exacto = brentq(residuo, 0.005, 0.10, xtol=1e-16, rtol=8.9e-16)
err_new = np.abs(np.array(hist_new) - f_exacto)
err_bis = np.abs(np.array(hist_bis) - f_exacto)
err_pf = np.abs(1 / np.array(hist_pf)**2 - f_exacto)

p_newton = orden_convergencia(err_new[:3])
p_biseccion = orden_convergencia(err_bis[:3])
comparar("orden observado de Newton", p_newton,
         LIBRO["colebrook_orden_newton"], 0.02, "")
print(f"orden observado de la bisección                {p_biseccion:.2f}")

if REVISAR_ORDEN:
    assert abs(p_newton - 2.0) < 0.02, "el orden de Newton debe valer dos"
    print("el orden observado confirma el Teorema 3.1")
else:
    print("complete la celda anterior y ponga REVISAR_ORDEN = True")

La Figura 3.3 del libro muestra el error absoluto en escala logarítmica
frente al número de iteración. La bisección describe una recta de
pendiente suave, el punto fijo una recta más inclinada y Newton una curva
que se dobla hacia abajo, firma inconfundible del orden cuadrático. La
celda siguiente la reproduce.

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(np.arange(1, err_bis.size + 1), np.maximum(err_bis, 1e-17),
            "o-", color=COLORES["gris"], ms=3.2, label="bisección")
ax.semilogy(np.arange(1, err_pf.size + 1), np.maximum(err_pf, 1e-17),
            "s-", color=COLORES["naranja"], ms=4.0, label="punto fijo")
ax.semilogy(np.arange(1, err_new.size + 1), np.maximum(err_new, 1e-17),
            "D-", color=COLORES["azul"], ms=4.5, label="Newton-Raphson")
ax.axhline(1e-10, color=COLORES["rojo"], lw=0.9, ls="--")
ax.text(9.0, 2.2e-10, "tolerancia 1e-10", color=COLORES["rojo"], fontsize=9)
ax.set_xlabel("iteración k")
ax.set_ylabel("error absoluto del factor de fricción")
ax.set_ylim(1e-17, 1e-1)
ax.set_title("Convergencia sobre la ecuación de Colebrook y White")
ax.legend(loc="upper right")
plt.show()

## 3. Una comprobación independiente

La verificación del Ejemplo 3.1 se apoya en la correlación explícita de
Swamee y Jain, que no es un sustituto del cálculo implícito sino una
comprobación de orden de magnitud. Su error del 0.6 por ciento se
traslada íntegro a la pérdida de carga y en una red mallada se acumula
tramo a tramo.

In [ ]:
f_swamee = 0.25 / (np.log10(rugosidad_relativa / 3.7
                            + 5.74 / reynolds**0.9))**2
desvio = 100 * (f_swamee - LIBRO["colebrook_factor_friccion"]) / LIBRO["colebrook_factor_friccion"]
comparar("factor de fricción de Swamee y Jain", f_swamee,
         LIBRO["colebrook_swamee_jain"], 1e-6, "")
print(f"apartamiento respecto del valor implícito      {desvio:.2f} por ciento")
assert abs(desvio - 0.61) < 0.02, "el apartamiento publicado es de 0.61 por ciento"

## 4. Sistemas acoplados y estructura del jacobiano

Un modelo estacionario realista rara vez tiene una sola incógnita. La
Figura 3.4 del libro muestra una cascada de balances con retromezcla
entre etapas contiguas y el patrón tridiagonal del jacobiano que resulta
de esa topología. La aplicación de referencia es un sistema de cinco
lagunas de estabilización de 2000 m³ que recibe 1200 m³/d con una demanda
bioquímica de oxígeno afluente de 250 mg/L, con un intercambio por
dispersión de 600 m³/d entre lagunas contiguas y cinética de saturación
de parámetros \(k_{\max}=60\) mg/(L d) y \(K_s=40\) mg/L.

El balance de la laguna interior es la Ecuación 3.6 del libro. Las
lagunas primera y última tienen un vecino menos y sus balances se
escriben aparte, tal como hace el Listado 3.2.

In [ ]:
from scipy.optimize import root

PARAMETROS = dict(Q=1200.0, Qr=600.0, V=2000.0, S0=250.0,
                  kmax=60.0, Ks=40.0)


def residuos_cascada(S: np.ndarray, par: dict) -> np.ndarray:
    """Residuos de los balances de la cascada de lagunas, en mg/d."""
    Q, Qr, V, S0 = par["Q"], par["Qr"], par["V"], par["S0"]
    r = par["kmax"] * S / (par["Ks"] + S)
    R = np.empty_like(S)
    R[0] = Q * S0 - (Q + Qr) * S[0] + Qr * S[1] - V * r[0]
    R[1:-1] = ((Q + Qr) * S[:-2] - (Q + 2 * Qr) * S[1:-1]
               + Qr * S[2:] - V * r[1:-1])
    R[-1] = (Q + Qr) * S[-2] - (Q + Qr) * S[-1] - V * r[-1]
    return R


def jacobiano_cascada(S: np.ndarray, par: dict) -> np.ndarray:
    """Jacobiano analítico, tridiagonal por la topología de la cascada."""
    Q, Qr, V = par["Q"], par["Qr"], par["V"]
    dr = par["kmax"] * par["Ks"] / (par["Ks"] + S)**2
    J = np.diag(-(Q + 2 * Qr) - V * dr)
    J += np.diag(np.full(S.size - 1, Q + Qr), -1)
    J += np.diag(np.full(S.size - 1, Qr), 1)
    J[0, 0] = -(Q + Qr) - V * dr[0]
    J[-1, -1] = -(Q + Qr) - V * dr[-1]
    return J


inicial = np.full(5, PARAMETROS["S0"])
con_jac = root(residuos_cascada, inicial, jac=jacobiano_cascada,
               args=(PARAMETROS,), tol=1e-12)
sin_jac = root(residuos_cascada, inicial, args=(PARAMETROS,), tol=1e-12)
perfil = con_jac.x

print("perfil de DBO por laguna, en mg/L")
print(np.round(perfil, 2))
print(f"con jacobiano   evaluaciones del residuo {con_jac.nfev:3d}"
      f"   del jacobiano {con_jac.njev:2d}")
print(f"sin jacobiano   evaluaciones del residuo {sin_jac.nfev:3d}")

In [ ]:
ok = [comparar(f"laguna {i + 1}", perfil[i], LIBRO[f"lagunas_perfil_{i + 1}"],
               5e-3, "mg/L") for i in range(5)]
remocion = 100 * (1 - perfil[-1] / PARAMETROS["S0"])
retencion = 5 * PARAMETROS["V"] / PARAMETROS["Q"]
ok.append(comparar("remoción global", remocion, LIBRO["lagunas_remocion"],
                   5e-3, "por ciento"))
ok.append(comparar("tiempo de retención total", retencion,
                   LIBRO["lagunas_retencion"], 5e-3, "d"))
assert all(ok), "el perfil no reproduce el que publica la sección 3.1.2"
assert (con_jac.nfev, con_jac.njev) == (19, 2), \
    "el conteo con jacobiano analítico debe ser 19 residuos y 2 jacobianos"
assert sin_jac.nfev == 29, "sin jacobiano el libro reporta 29 evaluaciones"
print("perfil, conteos y desempeño coinciden con el libro")

### Ejercicio 3

La verificación del sistema no se hace mirando la gráfica sino sumando
los cinco residuos, cuya suma se reduce al balance global
\(Q S_0 - Q S_n - V \sum_i r(S_i) = 0\). Complete la función que
devuelve la carga afluente, la efluente, el consumo y el cierre relativo.
El libro reporta 300000 mg/d, 8792.84 mg/d, 291207.16 mg/d y un cierre de
1.9 × 10⁻¹⁶, es decir precisión de máquina.

In [ ]:
# COMPLETE: arme el balance global de la cascada.
#   carga afluente  = Q*S0
#   carga efluente  = Q*S[-1]
#   consumo         = V*suma de kmax*S/(Ks+S)
#   cierre relativo = (afluente - efluente - consumo)/afluente
REVISAR_CIERRE = False


def cierre_global(S: np.ndarray, par: dict) -> dict:
    """Balance global de masa de la cascada, en mg/d."""
    return dict(afluente=0.0, efluente=0.0, consumo=0.0, cierre=1.0)

In [ ]:
balance = cierre_global(perfil, PARAMETROS)
comparar("carga afluente", balance["afluente"],
         LIBRO["lagunas_carga_afluente"], 1e-6, "mg/d")
comparar("carga efluente", balance["efluente"],
         LIBRO["lagunas_carga_efluente"], 5e-2, "mg/d")
comparar("consumo por reacción", balance["consumo"],
         LIBRO["lagunas_consumo"], 5e-2, "mg/d")
print(f"cierre relativo del balance global             {balance['cierre']:.3e}")

if REVISAR_CIERRE:
    assert abs(balance["cierre"]) < 1e-12, \
        "el balance global debe cerrar a precisión de máquina"
    print("el cierre confirma que el sistema algebraico se resolvió bien, "
          "no que el modelo sea adecuado")
else:
    print("complete la celda anterior y ponga REVISAR_CIERRE = True")

## 5. Newton amortiguado y punto inicial lejano

El Algoritmo 3.1 del libro reduce a la mitad el paso de Newton hasta que
la norma del residuo disminuya y hasta que el punto respete el dominio
físico. Con ese retroceso, un punto inicial de 1 mg/L, muy alejado del
correcto, alcanza la misma solución en seis iteraciones.

In [ ]:
def newton_amortiguado(F, J, x0, dominio, tol=1e-12, maxit=50,
                       lam_min=1e-4):
    """Newton con retroceso, según el Algoritmo 3.1 del libro."""
    x = np.array(x0, dtype=float)
    r = F(x)
    rho = np.max(np.abs(r))
    rho0 = rho
    historial = [rho]
    for k in range(1, maxit + 1):
        dx = np.linalg.solve(J(x), -r)
        lam = 1.0
        while True:
            x_mas = x + lam * dx
            r_mas = F(x_mas)
            if dominio(x_mas) and np.max(np.abs(r_mas)) < rho:
                break
            lam *= 0.5
            if lam < lam_min:
                return x, k, historial, False
        x, r = x_mas, r_mas
        rho = np.max(np.abs(r))
        historial.append(rho)
        if rho <= tol * rho0:
            return x, k, historial, True
    return x, maxit, historial, False


campo_valido = lambda S: bool(np.all(S > 0.0))
x_am, it_am, hist_am, exito = newton_amortiguado(
    lambda S: residuos_cascada(S, PARAMETROS),
    lambda S: jacobiano_cascada(S, PARAMETROS),
    np.full(5, 1.0), campo_valido)

print(f"iteraciones desde 1 mg/L   {it_am}   convergió {exito}")
print("solución", np.round(x_am, 2))
assert it_am == 6, "el libro reporta seis iteraciones desde 1 mg/L"
assert np.allclose(x_am, perfil, atol=1e-6), \
    "el Newton amortiguado debe llegar a la misma solución"
print("coincide con la sección 3.1.2 del libro")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.9))
etapas = np.arange(1, 6)
ax1.plot(etapas, perfil, "o-", color=COLORES["azul"], label="perfil calculado")
ax1.axhline(PARAMETROS["S0"], color=COLORES["gris"], ls="--", lw=0.9)
ax1.text(1.05, PARAMETROS["S0"] - 18, "afluente 250 mg/L",
         color=COLORES["gris"], fontsize=9)
ax1.set_xlabel("laguna")
ax1.set_ylabel("DBO (mg/L)")
ax1.set_xticks(etapas)
ax1.set_title("Perfil estacionario de la cascada")
ax1.legend()

ax2.semilogy(np.arange(len(hist_am)), np.maximum(hist_am, 1e-14), "o-",
             color=COLORES["rojo"])
ax2.set_xlabel("iteración")
ax2.set_ylabel("norma infinito del residuo (mg/d)")
ax2.set_title("Newton amortiguado desde 1 mg/L")
plt.tight_layout()
plt.show()

## 6. Del régimen permanente al transitorio

El régimen permanente describe la condición sostenida, pero muchas
preguntas se refieren al camino que conduce hasta ella. Cuando la
hipótesis de parámetros concentrados vale, el balance transitorio
conserva la acumulación y se convierte en un problema de valor inicial,
la Ecuación 3.7 del libro.

El Ejemplo 3.2 parte de 150 g/L de azúcares reductores y 0.50 g/L de
biomasa, con cinética de Monod de \(K_s = 1.2\) g/L y rendimiento
\(Y = 0.08\). A 30 °C la velocidad específica máxima vale 0.30 por hora.
Se pide el tiempo necesario para que el sustrato descienda a 1 g/L.

In [ ]:
from scipy.integrate import solve_ivp

KS_MONOD = 1.2        # g/L
RENDIMIENTO = 0.08    # g de biomasa por g de sustrato
X0, S0_LOTE = 0.50, 150.0    # g/L


def fermentador(t, y, mu_max):
    """Campo del fermentador en lote, con X y S en g/L."""
    X, S = y
    mu = mu_max * S / (KS_MONOD + S)
    return [mu * X, -mu * X / RENDIMIENTO]

### Ejercicio 4

Los eventos se declaran como funciones que se anulan en el instante
buscado. Escriba el evento que detecta el descenso del sustrato hasta
1 g/L, marque su dirección como descendente y hágalo terminal, de modo
que la integración se detenga allí. El Listado 3.3 del libro muestra la
estructura. La celda de partida vigila un umbral de 100 g/L, que se
alcanza mucho antes y produce un tiempo evidentemente falso.

In [ ]:
# COMPLETE: el evento debe anularse cuando el sustrato llega a 1 g/L.
# Marque agotamiento.direction = -1 y agotamiento.terminal = True.
REVISAR_EVENTO = False


def agotamiento(t, y, mu_max):
    """Se anula cuando el sustrato alcanza el umbral de interés."""
    return y[1] - 100.0        # marcador de posición


agotamiento.direction = -1
agotamiento.terminal = True

In [ ]:
solucion = solve_ivp(fermentador, (0.0, 30.0), [X0, S0_LOTE], args=(0.30,),
                     method="RK45", rtol=1e-9, atol=1e-11,
                     dense_output=True, events=agotamiento)
t_agotamiento = float(solucion.t_events[0][0])
invariante = solucion.y[0] + RENDIMIENTO * solucion.y[1]

comparar("tiempo de agotamiento a 30 grados", t_agotamiento,
         LIBRO["fermentador_tiempo_30C"], 1e-4, "h")
print(f"evaluaciones del campo                         {solucion.nfev:d}")
print(f"invariante X + Y S, desviación máxima          "
      f"{np.max(np.abs(invariante - 12.5)):.2e} g/L")

if REVISAR_EVENTO:
    assert abs(t_agotamiento - LIBRO["fermentador_tiempo_30C"]) < 1e-4, \
        "el instante no coincide con el Ejemplo 3.2"
    assert solucion.nfev == 584, \
        "con evento terminal el libro reporta 584 evaluaciones del campo"
    assert np.max(np.abs(invariante - 12.5)) < 1e-12, \
        "el invariante X + Y S debe conservarse"
    print("tiempo, costo e invariante coinciden con el Ejemplo 3.2")
else:
    print("complete la celda anterior y ponga REVISAR_EVENTO = True")

El invariante es la verificación exacta que ofrece este modelo. Sumando
el balance de biomasa con \(Y\) veces el de sustrato se obtiene
\(\mathrm{d}(X + Y S)/\mathrm{d}t = 0\), de modo que la cantidad
\(X + Y S\) vale 12.5 g/L durante todo el lote. El invariante reduce
además el sistema a una cuadratura en el sustrato, cuya evaluación de
alta precisión sirve de referencia independiente del integrador.

In [ ]:
from scipy.integrate import quad

CONSTANTE = X0 + RENDIMIENTO * S0_LOTE      # g/L


def tiempo_por_cuadratura(mu_max: float, s_final: float = 1.0) -> float:
    """Tiempo de agotamiento por cuadratura sobre el invariante."""
    integrando = lambda s: -(RENDIMIENTO * (KS_MONOD + s)
                             / (mu_max * s * (CONSTANTE - RENDIMIENTO * s)))
    valor, _ = quad(integrando, S0_LOTE, s_final, epsabs=1e-13,
                    epsrel=1e-13, limit=300)
    return float(valor)


t_cuadratura = tiempo_por_cuadratura(0.30)
comparar("tiempo por cuadratura a 30 grados", t_cuadratura,
         LIBRO["fermentador_tiempo_30C"], 5e-7, "h")
comparar("invariante X + Y S", CONSTANTE, LIBRO["fermentador_invariante"],
         1e-12, "g/L")
assert abs(t_cuadratura - LIBRO["fermentador_tiempo_30C"]) < 5e-7
print("la cuadratura de alta precisión confirma el resultado del integrador")

## 7. Sistemas distribuidos y método de líneas

Cuando el estado depende de la posición, el balance conduce a una
ecuación en derivadas parciales. La Definición 3.4 del libro llama método
de líneas a discretizar únicamente las derivadas espaciales sobre una
malla fija, con lo cual el problema se convierte en un sistema de
ecuaciones diferenciales ordinarias cuyas incógnitas son los valores
nodales. La Figura 3.9 recorre el procedimiento desde el estencil de tres
puntos hasta la matriz tridiagonal.

El Ejemplo 3.4 descarga 25 kg de un compuesto biodegradable en un río de
sección 18 m², velocidad media 0.35 m/s y coeficiente de dispersión
longitudinal 12 m²/s, con decaimiento de primer orden de 0.25 por día. La
solución analítica de una inyección instantánea en un canal infinito, la
Ecuación 3.16 del libro, verifica la implementación de manera concluyente.

In [ ]:
from scipy.sparse import diags

RIO = dict(u=0.35, D=12.0, k=0.25 / 86400, M=25.0, area=18.0, x0=1200.0)
LARGO, N_INTERVALOS = 5000.0, 250
T_INICIAL, T_FINAL = 600.0, 3600.0


def perfil_exacto(x, t, u, D, k, M, area, x0):
    """Solución analítica de una inyección instantánea, en kg/m3."""
    s2 = 4 * D * t
    return (M / (area * np.sqrt(np.pi * s2))
            * np.exp(-(x - x0 - u * t)**2 / s2) * np.exp(-k * t))


malla = np.linspace(0, LARGO, N_INTERVALOS + 1)
x_int = malla[1:-1]                  # nodos interiores, Dirichlet nula
dx = malla[1] - malla[0]             # m
n_nodos = x_int.size
peclet = RIO["u"] * dx / RIO["D"]
comparar("número de Péclet de celda", peclet, LIBRO["rio_peclet_celda"],
         5e-4, "")
print(f"paso espacial {dx:.1f} m con {n_nodos} nodos interiores")

### Ejercicio 5

Complete el ensamblaje de la matriz de diferencias finitas del Listado
3.5. La subdiagonal recoge la difusión más la mitad de la advección, la
diagonal el término difusivo y el decaimiento, y la superdiagonal la
difusión menos la mitad de la advección. La celda de partida solo arma la
difusión y el decaimiento, de modo que la pluma no viaja aguas abajo y el
error contra la solución analítica resulta enorme.

In [ ]:
# COMPLETE: agregue la advección centrada a la subdiagonal y a la
# superdiagonal, con signo +u/(2*dx) abajo y -u/(2*dx) arriba.
REVISAR_MATRIZ = False


def matriz_transporte(n: int, dx: float, u: float, D: float, k: float):
    """Diferencias centradas para advección, difusión y decaimiento."""
    sub = np.full(n - 1, D / dx**2)          # falta la advección
    dia = np.full(n, -2 * D / dx**2 - k)
    sup = np.full(n - 1, D / dx**2)          # falta la advección
    return diags([sub, dia, sup], [-1, 0, 1], format="csc")

In [ ]:
A = matriz_transporte(n_nodos, dx, RIO["u"], RIO["D"], RIO["k"])
c_inicial = perfil_exacto(x_int, T_INICIAL, **RIO)
marcha = solve_ivp(lambda t, c: A @ c, (T_INICIAL, T_FINAL), c_inicial,
                   method="BDF", jac=lambda t, c: A,
                   rtol=1e-11, atol=1e-16, t_eval=[T_FINAL])
c_final = marcha.y[:, 0]
c_exacto = perfil_exacto(x_int, T_FINAL, **RIO)

error_max = float(np.max(np.abs(c_final - c_exacto)))
masa = float(trapecio(c_final, x_int) * RIO["area"])
pico_num = 1e3 * c_final.max()
pico_ana = 1e3 * c_exacto.max()
abscisa = x_int[int(np.argmax(c_final))]

print(f"pico numérico            {pico_num:.4f} mg/L en x = {abscisa:.0f} m")
print(f"pico analítico           {pico_ana:.4f} mg/L")
comparar("pico analítico en x igual a 2460 m", pico_ana,
         LIBRO["rio_pico_analitico"], 1e-3, "mg/L")
comparar("abscisa del pico", abscisa, LIBRO["rio_abscisa_pico"], 1e-9, "m")
comparar("error máximo frente a la analítica", error_max,
         LIBRO["rio_error_maximo"], 5e-8, "kg/m3")
comparar("masa remanente", masa, LIBRO["rio_masa_remanente"], 1e-6, "kg")

if REVISAR_MATRIZ:
    assert abs(masa - LIBRO["rio_masa_remanente"]) < 1e-6, \
        "la masa remanente debe coincidir con M exp(-k t)"
    assert error_max < 1e-5, "el error contra la analítica es demasiado grande"
    print("el perfil y la masa remanente coinciden con el Ejemplo 3.4")
else:
    print("complete la celda anterior y ponga REVISAR_MATRIZ = True")

Conviene señalar una diferencia de lectura. El Ejemplo 3.4 atribuye el
pico de 1.8655 mg/L a la integración numérica, y ese número es en
realidad el máximo de la solución analítica en la abscisa 2460 m. El
perfil calculado con paso de 20 m entrega 1.8663 mg/L en el mismo nodo,
ocho diezmilésimas de miligramo por litro por encima, muy dentro del
error máximo de 7.18 × 10⁻⁶ kg/m³ que el propio ejemplo reporta. El
cuaderno verifica ambos valores por separado para que la diferencia
quede a la vista.

In [ ]:
fig, ax = plt.subplots()
finos = np.linspace(0, LARGO, 1200)
ax.plot(finos / 1000, 1e3 * perfil_exacto(finos, T_INICIAL, **RIO),
        color=COLORES["gris"], lw=1.0, label="analítica, t = 600 s")
ax.plot(finos / 1000, 1e3 * perfil_exacto(finos, T_FINAL, **RIO),
        color=COLORES["azul"], lw=1.3, label="analítica, t = 3600 s")
ax.plot(x_int[::8] / 1000, 1e3 * c_final[::8], "o", color=COLORES["rojo"],
        ms=3.6, mfc="none", label="método de líneas, dx = 20 m")
ax.set_xlim(0.6, 4.0)
ax.set_xlabel("abscisa x (km)")
ax.set_ylabel("concentración c (mg/L)")
ax.set_title("Transporte de un vertimiento, una hora después de la descarga")
ax.legend()
plt.show()

## 8. Problemas del capítulo

El problema 3-6 pide resolver Colebrook y White para una tubería de PVC
de 0.20 m de diámetro y rugosidad 0.0015 mm que conduce 45 L/s de agua a
25 °C, con viscosidad cinemática de 0.893 × 10⁻⁶ m²/s, partiendo de
\(f_0 = 0.02\) y con tolerancia de 10⁻¹⁰. El problema 3-7 plantea el
residuo del balance de sal de un tanque de mezcla que recibe salmueras de
12 g/L y 35 g/L con caudales de 4 m³/h y 7 m³/h, y pide verificar el
cierre global de masa. Los dos se resuelven aquí con las funciones que ya
están escritas.

In [ ]:
# Problema 3-6
D_pvc, eps_pvc, Q_pvc, nu_pvc = 0.20, 0.0015e-3, 0.045, 0.893e-6
V_pvc = Q_pvc / (np.pi * D_pvc**2 / 4)
Re_pvc = V_pvc * D_pvc / nu_pvc
rug_pvc = eps_pvc / D_pvc
res_pvc = lambda f: residuo_colebrook(f, Re_pvc, rug_pvc)
der_pvc = lambda f: ((residuo_colebrook(f + 1e-8, Re_pvc, rug_pvc)
                      - residuo_colebrook(f - 1e-8, Re_pvc, rug_pvc)) / 2e-8)
f_pvc, it_pvc, ev_pvc, _ = newton(res_pvc, der_pvc, 0.02)
print(f"problema 3-6   Re = {Re_pvc:,.0f}   f = {f_pvc:.6f}"
      f"   iteraciones {it_pvc}")

# Problema 3-7
def residuo_mezcla(C, Q1=4.0, C1=12.0, Q2=7.0, C2=35.0):
    """Residuo del balance de sal del tanque, en g/h."""
    return Q1 * C1 * 1e3 + Q2 * C2 * 1e3 - (Q1 + Q2) * C * 1e3


C_mezcla = brentq(residuo_mezcla, 0.0, 100.0, xtol=1e-14)
entra = (4.0 * 12.0 + 7.0 * 35.0) * 1e3
sale = 11.0 * C_mezcla * 1e3
print(f"problema 3-7   concentración de salida {C_mezcla:.4f} g/L")
print(f"               cierre de masa {(entra - sale) / entra:.2e}")
assert abs(C_mezcla - (4 * 12 + 7 * 35) / 11) < 1e-10

## 9. Cierre

Al terminar este cuaderno el estudiante debe poder hacer lo siguiente.

1. Escribir el residuo de un balance estacionario y elegir entre Brent,
   Newton y punto fijo con un argumento de costo y de robustez, no de
   gusto. Si algo no salió, revise la Tabla 3.1 y el Teorema 3.1.
2. Estimar el orden de convergencia observado y usarlo como detector de
   errores en la derivada. Un método que debería ser cuadrático y exhibe
   orden uno casi siempre tiene la derivada mal escrita.
3. Resolver un sistema acoplado con el jacobiano analítico y verificar la
   solución con un balance global. Si el cierre no baja de 10⁻¹², el
   residuo o el jacobiano tienen un error de signo.
4. Integrar un problema de valor inicial con evento terminal y verificar
   el resultado con un invariante del modelo o con una cuadratura de alta
   precisión. Revise el Ejemplo 3.2 si el invariante se degrada.
5. Ensamblar la matriz del método de líneas y comparar el perfil contra
   la solución analítica. Revise la Definición 3.4 y el Algoritmo 3.2 si
   el error no baja del uno por ciento del pico.

El Capítulo 3 continúa con el comportamiento del sistema bajo condiciones
distintas, que es el asunto del cuaderno siguiente.